# 04 — Hyperparameter Tuning (Optuna)

This notebook uses **Optuna** to tune:
1. Random Forest
2. XGBoost

Objective: Maximise stratified 5-fold cross-validated ROC-AUC.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier

from src.data_loader import load_cleaned_data
from src.features import engineer_features
from src.train import save_model
from src.utils import split_data, setup_logging, save_json, get_reports_dir

setup_logging()
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 50  # reduce for faster runs

In [ ]:
# Prepare data
df = load_cleaned_data()
df = engineer_features(df)
X_train, X_test, y_train, y_test = split_data(df)
print(f'Training shape: {X_train.shape}')

## 1. Tune Random Forest

In [ ]:
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 4, 16),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
    }
    model = RandomForestClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

rf_study = optuna.create_study(direction='maximize', study_name='RF_tuning')
rf_study.optimize(rf_objective, n_trials=N_TRIALS)

print(f'\nBest RF AUC: {rf_study.best_value:.4f}')
print(f'Best RF params: {rf_study.best_params}')

## 2. Tune XGBoost

In [ ]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'use_label_encoder': False,
        'eval_metric': 'logloss',
        'random_state': 42,
        'n_jobs': -1,
    }
    model = XGBClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

xgb_study = optuna.create_study(direction='maximize', study_name='XGB_tuning')
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS)

print(f'\nBest XGB AUC: {xgb_study.best_value:.4f}')
print(f'Best XGB params: {xgb_study.best_params}')

## 3. Compare & Save Best

In [ ]:
# Determine overall best
if xgb_study.best_value >= rf_study.best_value:
    print('XGBoost is the best model')
    best_params = xgb_study.best_params
    best_params.update({'use_label_encoder': False, 'eval_metric': 'logloss',
                        'random_state': 42, 'n_jobs': -1})
    best_model = XGBClassifier(**best_params)
    best_name = 'XGBoost_tuned'
else:
    print('Random Forest is the best model')
    best_params = rf_study.best_params
    best_params.update({'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1})
    best_model = RandomForestClassifier(**best_params)
    best_name = 'RandomForest_tuned'

# Train on full training set
best_model.fit(X_train, y_train)

# Save
save_model(best_model)

# Save best hyperparameters
save_json({
    'best_model': best_name,
    'best_params': best_params,
    'rf_best_auc': rf_study.best_value,
    'xgb_best_auc': xgb_study.best_value,
}, os.path.join(get_reports_dir(), 'best_hyperparameters.json'))

print(f'\n✅ Saved tuned {best_name} to models/model.pkl')

## 4. Optuna Visualisation

In [ ]:
try:
    import optuna.visualization as ov
    fig = optuna.visualization.plot_optimization_history(xgb_study)
    fig.show()
except ImportError:
    print('Install plotly for Optuna visualisations: pip install plotly')